# 🩺 Huấn luyện Mamba-UNet (MambaVision Backbone) trên Google Colab

Quy trình chuẩn hóa toàn diện:
1. **Kiểm tra phần cứng GPU** (`!nvidia-smi`)
2. **Cài đặt môi trường PyTorch 2.6.0 (cu126) & Triton 3.3.1** (tương thích chính xác với C++ CUDA ABI=True)
3. **Mount Google Drive & Cài đặt 2 Wheels CUDA (`causal_conv1d` + `mamba_ssm`)** từ Drive
4. **Khởi động lại Runtime** để nạp kernel C++/CUDA vào hệ thống
5. **Kiểm tra xác minh CUDA Selective Scan** (fp32 + fp16, forward + backward)
6. **Đồng bộ mã nguồn từ GitHub**
7. **Giải nén dữ liệu từ Google Drive vào SSD cục bộ** (có tùy chọn số lượng ca bệnh nhân `NUM_PATIENTS`)
8. **Huấn luyện mô hình với đầy đủ tham số tùy chỉnh** (Backbone, Pretrained, Activation, Loss, LR, Workers, Resume...)
9. **Đánh giá 3D cấp độ bệnh nhân** trên tập Validation
10. **Lưu kết quả & Checkpoints vào Google Drive theo từng thư mục phiên chạy kèm bản báo cáo Markdown/JSON** (tự động append nếu Resume)

In [1]:
# 1. Kiểm tra phần cứng và GPU khả dụng
!nvidia-smi

Sat Nov 14 12:29:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Thiết lập môi trường PyTorch 2.6.0 (cu126) + Triton 3.3.1
# (Chạy ở tiến trình riêng, không import torch trực tiếp trong kernel này)
import subprocess, sys

def probe(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

TORCH_PROBE = "import torch;print(torch.__version__, int(torch._C._GLIBCXX_USE_CXX11_ABI))"
cur = probe(TORCH_PROBE)
print("Torch hiện tại:", cur)

if cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"):
    print("⏭️ Đã có torch 2.6.0+cu126 (ABI=True), bỏ qua bước cài torch.")
else:
    print("🧹 Dọn dẹp và cài đặt PyTorch 2.6.0+cu126...")
    !rm -rf /usr/local/lib/python3.13/dist-packages/~* 2>/dev/null || true
    !pip uninstall torch torchvision torchaudio causal-conv1d mamba-ssm -y -q 2>/dev/null || true
    !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126 -q
    cur = probe(TORCH_PROBE)
    print("Torch sau khi cài:", cur)

assert cur and cur.startswith("2.6.0+cu126") and cur.endswith(" 1"), \
    f"❌ Torch chưa đúng (cần 2.6.0+cu126 và ABI=1, đang là: {cur}). Hãy thử chạy lại cell này."

print("\n⚙️ Cài đặt Triton 3.3.1 và các thư viện cần thiết (SimpleITK, etc.)...")
!python -m pip install triton==3.3.1 SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib -q
trt = probe("import triton;print(triton.__version__, hasattr(triton,'set_allocator'))")
print("Triton:", trt)
assert trt and trt.endswith("True"), f"❌ Triton chưa có set_allocator (đang là: {trt})"

print("\n✅ CELL 2 XONG. Chạy tiếp cell 3 để mount Drive và cài 2 wheels.")

Torch hiện tại: 2.14.1+cu130 1
🧹 Dọn dẹp và cài đặt PyTorch 2.6.0+cu126...
Torch sau khi cài: 2.6.0+cu126 1

⚙️ Cài đặt Triton 3.3.1 và các thư viện cần thiết (SimpleITK, etc.)...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu126 requires triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 3.3.1 which is incompatible.
Triton: 3.3.1 True

✅ CELL 2 XONG. Chạy tiếp cell 3 để mount Drive và cài 2 wheels.


In [3]:
# 3. Mount Google Drive & Cài đặt 2 Wheel CUDA (chọn theo ABI thật)
import os, subprocess, sys
from google.colab import drive

print("🔗 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Dò tìm thư mục wheels trên Google Drive (hỗ trợ cả cấu trúc mirror GG_1):
WHEEL_CANDIDATES = [
    "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels",
    "/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels",
    "/content/drive/MyDrive/wheels",
]
WHEEL_DIR = None
for d in WHEEL_CANDIDATES:
    if os.path.isdir(d):
        WHEEL_DIR = d
        print(f"🎯 Đã tìm thấy thư mục wheels tại: {WHEEL_DIR}")
        break

assert WHEEL_DIR is not None, "❌ Không tìm thấy thư mục wheels trên Google Drive! Hãy đảm bảo thư mục wheels đã được tải lên Drive."

r = subprocess.run([sys.executable, "-c",
                    "import torch;print('TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')"],
                   capture_output=True, text=True)
abi = r.stdout.strip()
assert abi in ("TRUE", "FALSE"), f"❌ Không đọc được ABI: {r.stderr}"
print(f"🔍 ABI của torch đã cài: cxx11abi{abi}")

CAUSAL_WHL = f"{WHEEL_DIR}/causal_conv1d-1.7.0+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"
MAMBA_WHL  = f"{WHEEL_DIR}/mamba_ssm-2.3.2.post1+cu12torch2.6cxx11abi{abi}-cp313-cp313-linux_x86_64.whl"

for p in (CAUSAL_WHL, MAMBA_WHL):
    assert os.path.exists(p), f"❌ Không thấy wheel: {p}"

print("\n🚀 Đang cài 2 wheel từ Drive...")
!python -m pip install "{CAUSAL_WHL}" --force-reinstall --no-deps -q
!python -m pip install "{MAMBA_WHL}" --force-reinstall --no-deps -q

print("\n✅ CELL 3 XONG. Chạy tiếp cell 4 (khởi động lại runtime).")

🔗 Đang kết nối Google Drive...
Mounted at /content/drive
🎯 Đã tìm thấy thư mục wheels tại: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/wheels
🔍 ABI của torch đã cài: cxx11abiTRUE

🚀 Đang cài 2 wheel từ Drive...

✅ CELL 3 XONG. Chạy tiếp cell 4 (khởi động lại runtime).


In [ ]:
# 4. Khởi động lại Runtime để nạp thư viện C++/CUDA mới vào Python kernel
# ⚠️ LƯU Ý QUAN TRỌNG:
# Khi chạy ô này, Google Colab sẽ hiện thông báo đỏ: 'Your session crashed and has recovered'.
# Đây là điều HOÀN TOÀN BÌNH THƯỜNG và cần thiết để khởi động lại kernel nạp C++ CUDA extensions.
# Bạn chỉ cần ấn chạy tiếp Cell 5 ngay sau đó!
import os
os.kill(os.getpid(), 9)

In [1]:
# 5. Kiểm tra causal_conv1d + mamba_ssm (xuôi + ngược, fp32 + fp16)
import sys, traceback, torch

print("python :", f"cp{sys.version_info.major}{sys.version_info.minor}")
print("torch  :", torch.__version__, "| ABI =", torch._C._GLIBCXX_USE_CXX11_ABI)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
import triton
print("triton :", triton.__version__, "| set_allocator:", hasattr(triton, "set_allocator"))

ok = {}

# --- Causal-Conv1d ---
try:
    import causal_conv1d, causal_conv1d_cuda
    from causal_conv1d import causal_conv1d_fn
    for dt in (torch.float32, torch.float16):
        x = torch.randn(2, 16, 64, device="cuda", dtype=dt, requires_grad=True)
        w = torch.randn(16, 4, device="cuda", dtype=dt, requires_grad=True)
        y = causal_conv1d_fn(x, w, activation="silu")
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(x.grad).all()
    ok["causal_conv1d"] = True
    print("causal_conv1d : ✅ xuôi + ngược, fp32 + fp16")
except Exception:
    ok["causal_conv1d"] = False
    print("causal_conv1d : ❌"); traceback.print_exc()

# --- Mamba-SSM ---
try:
    import mamba_ssm, selective_scan_cuda
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    B, D, L, N = 2, 16, 64, 8
    for dt in (torch.float32, torch.float16):
        u     = torch.randn(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        delta = torch.rand(B, D, L, device="cuda", dtype=dt, requires_grad=True)
        A     = (-torch.rand(D, N, device="cuda")).requires_grad_()
        Bm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Cm    = torch.randn(B, N, L, device="cuda", dtype=dt, requires_grad=True)
        Dv    = torch.randn(D, device="cuda")
        y = selective_scan_fn(u, delta, A, Bm, Cm, D=Dv, delta_softplus=True)
        y.float().sum().backward()
        assert torch.isfinite(y).all() and torch.isfinite(u.grad).all()
    ok["mamba_ssm"] = True
    print("mamba_ssm     : ✅ selective_scan xuôi + ngược, fp32 + fp16")
except Exception:
    ok["mamba_ssm"] = False
    print("mamba_ssm     : ❌"); traceback.print_exc()

assert all(ok.values()), "❌ Kiểm tra mamba_ssm thất bại! Vui lòng kiểm tra lại log."
print("\n==> 🎉 MÔI TRƯỜNG CUDA MAMBA-SSM ĐÃ HOÀN TOÀN SẴN SÀNG!")

python : cp313
torch  : 2.6.0+cu126 | ABI = True
GPU    : NVIDIA L4
triton : 3.3.1 | set_allocator: True
causal_conv1d : ✅ xuôi + ngược, fp32 + fp16
mamba_ssm     : ✅ selective_scan xuôi + ngược, fp32 + fp16

==> 🎉 MÔI TRƯỜNG CUDA MAMBA-SSM ĐÃ HOÀN TOÀN SẴN SÀNG!


In [2]:
# 6. Tải và đồng bộ mã nguồn từ GitHub
import os, sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('mamba_unet') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

📥 Đang clone repository từ GitHub (nhánh main)...
Cloning into '/content/CT208E_LIDC_DuneAI_MambaVision'...
remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (356/356), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 356 (delta 175), reused 293 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (356/356), 12.40 MiB | 20.78 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/CT208E_LIDC_DuneAI_MambaVision
🚀 Thư mục làm việc hiện tại: /content/CT208E_LIDC_DuneAI_MambaVision


In [ ]:
# cell dùng để pull code về nếu có chỉnh sửa
%cd /content/CT208E_LIDC_DuneAI_MambaVision
!git pull origin main

/content/CT208E_LIDC_DuneAI_MambaVision
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 597 bytes | 298.00 KiB/s, done.
From https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision
 * branch            main       -> FETCH_HEAD
   1138001..522eacf  main       -> origin/main
Updating 1138001..522eacf
Fast-forward
 mamba_unet/backbone/pretrained.py | 10 +++++++---
 1 file changed, 7 insertions(+), 3 deletions(-)


In [3]:
# 7. Chuẩn bị dữ liệu từ Google Drive vào SSD cục bộ (/content/data/cache_1mm)
# Thư mục GG_1 trên máy tính được mirror trực tiếp vào Google Drive.
# Mã sẽ tự động tìm kiếm file zip theo đúng cấu trúc mirror của bạn:
import os
from common.data_pipeline.runtime_data import run_colab

DRIVE_ZIP_CANDIDATES = [
    # 1. Mirror trực tiếp thư mục GG_1 vào MyDrive:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 2. Mirror có thư mục cha GG_1 trong MyDrive:
    '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 3. File zip tại thư mục KAGGLE_UPLOAD:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/KAGGLE_UPLOAD/cache_1mm.zip',
    # 4. File zip đặt ngay tại gốc MyDrive:
    '/content/drive/MyDrive/cache_1mm.zip',
]

DRIVE_ZIP_PATH = None
for cand in DRIVE_ZIP_CANDIDATES:
    if os.path.exists(cand):
        DRIVE_ZIP_PATH = cand
        print(f'🎯 Đã tìm thấy file zip: {DRIVE_ZIP_PATH}')
        break

if DRIVE_ZIP_PATH is None:
    print('⚠️ Chưa thấy ở các đường dẫn cố định, sẽ tự động dò tìm sâu dưới /content/drive...')
    DRIVE_ZIP_PATH = 'auto'

# NUM_PATIENTS:
# - 0   : Giải nén toàn bộ 1010 bệnh nhân (cần ~40-60GB SSD Colab).
# - > 0 : Chọn N bệnh nhân đầu tiên (vd: 50, 100, 200) để huấn luyện nhanh và tiết kiệm dung lượng đĩa.
NUM_PATIENTS = 500

cache_dir = run_colab(
    zip_path=DRIVE_ZIP_PATH,
    dest_dir='/content/data/cache_1mm',
    n_patients=NUM_PATIENTS,
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

🎯 Đã tìm thấy file zip: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip
📦 Colab Data Pipeline: Chọn 500/1010 bệnh nhân từ cache_1mm.zip
CACHE_DIR=/content/data/cache_1mm
✅ Dữ liệu sẵn sàng tại: /content/data/cache_1mm


In [4]:
# 1. Tạo thư mục và copy thẳng file 364MB từ Drive sang để hệ thống nạp ngay trong 0.1 giây
!mkdir -p /content/CT208E_LIDC_DuneAI_MambaVision/pretrained_weights
!cp -f "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/pretrained_weights/"* /content/CT208E_LIDC_DuneAI_MambaVision/pretrained_weights/ 2>/dev/null || true
!cp -f "/content/drive/MyDrive/pretrained_weights/"* /content/CT208E_LIDC_DuneAI_MambaVision/pretrained_weights/ 2>/dev/null || true

# 2. Kiểm tra xem file đã sẵn sàng chưa
!ls -lh /content/CT208E_LIDC_DuneAI_MambaVision/pretrained_weights/

total 365M
-rw------- 1 root root 365M Nov 14 12:38 mamba_vision_t_nvidia_pretrained.pth.tar


In [5]:
# 8. Cấu hình chi tiết & Bắt đầu huấn luyện mô hình Mamba-UNet
import os

# =============================================================================
# BẢNG THAM SỐ TÙY CHỈNH (Tùy chỉnh trực tiếp các giá trị bên dưới)
# =============================================================================
# 1. Kiến trúc mô hình & Backbone MambaVision
BACKBONE = "mamba_vision_T"          # Lựa chọn: 'mamba_vision_T', 'mamba_vision_T2', 'mamba_vision_S', 'mamba_vision_B'
PRETRAINED = True                   # True: Dùng trọng số ImageNet-1K pretrained; False: Khởi tạo ngẫu nhiên
FREEZE_BACKBONE_EPOCHS = 3          # Số epoch đầu đóng băng backbone (-1: đóng băng suốt, 0: không đóng băng)
BACKBONE_LR_MULT = 0.1              # Hệ số nhân learning rate cho backbone so với decoder

# 2. Hàm kích hoạt Decoder (Activation)
# Lựa chọn: 'relu', 'gelu', 'silu', 'mish', 'leaky_relu', 'star_relu', 'soft_clamp_relu', 'soft_clamp_silu', 'identity'
ACTIVATION = "relu"

# 3. Hàm mất mát (Loss Function)
# Lựa chọn: 'dice_bce', 'dice', 'bce', 'tversky', 'focal_tversky', 'focal'
LOSS = "dice_bce"
LOSS_PARAMS = '{"bce_weight": 0.3, "dice_weight": 0.7, "pos_weight": 4.0}'  # R0/1a/1b: pos_weight=4.0 | Run 1c: thử pos_weight=8.0
# LOSS_PARAMS = '{"alpha": 0.3, "beta": 0.7}' # dành cho loss là tversky, beta lớn là sẽ chống bỏ sót

# 4. Siêu tham số huấn luyện (Training Hyperparameters)
EPOCHS = 50                         # Tổng số epoch huấn luyện
BATCH_SIZE = 8                      # Kích thước batch (khuyến nghị 8 trên Colab T4 16GB)
LR = 1e-4                           # Tốc độ học cơ sở (Learning rate)
WEIGHT_DECAY = 1e-4                 # Hệ số suy giảm trọng số AdamW (mặc định R0: 1e-4)
SCHEDULER = "cosine"                # Lựa chọn: 'cosine', 'plateau', 'step', 'none'
GRAD_CLIP = 1.0                     # Ngưỡng kẹp gradient tối đa (0 = tắt)
PATIENCE = 12                       # Số epoch chờ Early Stopping
WORKERS = 4                         # Số luồng nạp dữ liệu DataLoader (Colab khuyến nghị 2-4)

# 5. Quản lý phiên chạy & Khôi phục (Resume)
# RESUME: None (chạy mới từ đầu), 'auto' (tiếp tục phiên gần nhất bị ngắt), hoặc đường dẫn checkpoint/run cụ thể
RESUME = None                       # vd: 'auto' hoặc 'mamba_unet/runs/run_xxx/last_model.pth'
INIT_WEIGHTS = None                 # Nạp trọng số ban đầu nhưng tạo run mới (None nếu không dùng)
RUN_NAME = "mamba_colab_500item_50epoch_R1b_NegDynamic"         # Tên phiên chạy (để trống '' để tự sinh theo timestamp)
TAG = "colab_mamba"                 # Thẻ phân loại thí nghiệm
NOTES = "Huấn luyện Mamba-UNet với kernel CUDA native selective_scan trên Colab"
SEED = 42

# 6. Tham số Dữ liệu & Thí nghiệm Run 1 (R0 vs Run 1a / 1b / 1c)
CACHE_DIR = "/content/data/cache_1mm"            # Thư mục cache .npy (tái sử dụng cho các run)
SLICES_CSV = "/content/data/cache_1mm/slices_r1b_dynneg.csv"  # File index riêng cho từng run (để trống '' để dùng <CACHE_DIR>/slices.csv)
REBUILD_INDEX = False               # True: Ép dựng lại SLICES_CSV nếu đã có; False: Tự dựng nếu file chưa tồn tại
IMAGE_SIZE = 448                    # DUNEAI_IMAGE_SIZE: R0 = 512 | Run 1a / 1b / 1c = 448
MIN_TRAIN_AREA = 5.0                # --min-train-area (mm²): R0 = 20.0 | Run 1a / 1b / 1c = 5.0
NEG_RATIO = 0.5                    # --neg-ratio: R0 & Run 1a = 0.25 | Run 1b / 1c = 0.50
MIN_NEG_PER_CASE = 4               # R0/1a=1 | R1b/1c=4: tăng sàn cho ca không u/ít u (Run 1a bị FP 10/11 ca sạch) mà không làm phình ca >=8 lát u (0.5*8=4)
DYNAMIC_NEG = True                 # --dynamic-neg: R0 & Run 1a = False | Run 1b / 1c = True
KEEP_ALL_TRAIN_NEG = True          # --keep-all-train-neg: R0 & Run 1a = False | Run 1b / 1c = True

# 7. Tối ưu phần cứng & Tùy chọn phụ
FORCE_CUDA_BACKEND = True           # Ép dùng kernel CUDA native (DUNEAI_MAMBA_BACKEND=cuda)
USE_AMP = True                      # Bật Automatic Mixed Precision fp16 tăng tốc
GPU_WINDOW = False                  # Chuyển HU int16 lên GPU chuẩn hóa cửa sổ
NO_LUNG_MASK = False                # True: Không áp mặt nạ phổi ngoài lát cắt
NO_AUGMENT = False                  # True: Tắt lật/xoay ngẫu nhiên
LIMIT_TRAIN = 0                     # Giới hạn số lát cắt train (0 = toàn bộ)
LIMIT_VAL = 0                       # Giới hạn số lát cắt val (0 = toàn bộ)

# =============================================================================
# TỔ HỢP LỆNH HUẤN LUYỆN
# =============================================================================
env_vars = [f"DUNEAI_IMAGE_SIZE={IMAGE_SIZE}"]
if FORCE_CUDA_BACKEND:
    env_vars.append("DUNEAI_MAMBA_BACKEND=cuda")
env_prefix = " ".join(env_vars) + " "

cmd_parts = [
    f"{env_prefix}python -u -m mamba_unet.train_main train",
    f"--cache-dir '{CACHE_DIR}'",
    f"--min-train-area {MIN_TRAIN_AREA}",
    f"--neg-ratio {NEG_RATIO}",
    f"--min-neg-per-case {MIN_NEG_PER_CASE}",
    f"--backbone {BACKBONE}",
    f"{'--pretrained' if PRETRAINED else '--no-pretrained'}",
    f"--freeze-backbone-epochs {FREEZE_BACKBONE_EPOCHS}",
    f"--backbone-lr-mult {BACKBONE_LR_MULT}",
    f"--activation {ACTIVATION}",
    f"--loss {LOSS}",
    f"--epochs {EPOCHS}",
    f"--batch-size {BATCH_SIZE}",
    f"--lr {LR}",
    f"--weight-decay {WEIGHT_DECAY}",
    f"--scheduler {SCHEDULER}",
    f"--grad-clip {GRAD_CLIP}",
    f"--patience {PATIENCE}",
    f"--workers {WORKERS}",
    f"--seed {SEED}",
]
if SLICES_CSV and SLICES_CSV.strip():
    cmd_parts.append(f"--slices-csv '{SLICES_CSV.strip()}'")
if REBUILD_INDEX:
    cmd_parts.append("--rebuild-index")
if DYNAMIC_NEG:
    cmd_parts.append("--dynamic-neg")
if KEEP_ALL_TRAIN_NEG:
    cmd_parts.append("--keep-all-train-neg")
if LOSS_PARAMS and LOSS_PARAMS.strip():
    cmd_parts.append(f"--loss-params '{LOSS_PARAMS.strip()}'")
if RESUME:
    cmd_parts.append(f"--resume {RESUME}")
if INIT_WEIGHTS:
    cmd_parts.append(f"--init-weights '{INIT_WEIGHTS}'")
if RUN_NAME:
    cmd_parts.append(f"--run-name '{RUN_NAME}'")
if TAG:
    cmd_parts.append(f"--tag '{TAG}'")
if NOTES:
    cmd_parts.append(f"--notes '{NOTES}'")
if not USE_AMP:
    cmd_parts.append("--no-amp")
if GPU_WINDOW:
    cmd_parts.append("--gpu-window")
if NO_LUNG_MASK:
    cmd_parts.append("--no-lung-mask")
if NO_AUGMENT:
    cmd_parts.append("--no-augment")
if LIMIT_TRAIN > 0:
    cmd_parts.append(f"--limit-train {LIMIT_TRAIN}")
if LIMIT_VAL > 0:
    cmd_parts.append(f"--limit-val {LIMIT_VAL}")

# Tự động đồng bộ thời gian thực sang Google Drive sau mỗi Epoch (chống mất mát khi Colab ngắt)
MIRROR_SOURCES_DIR = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/DuneAI_Backups'

if os.path.exists(MIRROR_SOURCES_DIR):
    DRIVE_SYNC_DIR = f"{MIRROR_SOURCES_DIR}/mamba_unet/runs"
    cmd_parts.append(f"--drive-sync-dir '{DRIVE_SYNC_DIR}'")

full_train_cmd = " \\\n    ".join(cmd_parts)
print("🚀 Đang khởi động tiến trình huấn luyện...")
print(f"Lệnh thực thi:\n{full_train_cmd}\n" + "=" * 80)
!{full_train_cmd}

🚀 Đang khởi động tiến trình huấn luyện...
Lệnh thực thi:
DUNEAI_IMAGE_SIZE=448 DUNEAI_MAMBA_BACKEND=cuda python -u -m mamba_unet.train_main train \
    --cache-dir '/content/data/cache_1mm' \
    --min-train-area 5.0 \
    --neg-ratio 0.5 \
    --min-neg-per-case 4 \
    --backbone mamba_vision_T \
    --pretrained \
    --freeze-backbone-epochs 3 \
    --backbone-lr-mult 0.1 \
    --activation relu \
    --loss dice_bce \
    --epochs 50 \
    --batch-size 8 \
    --lr 0.0001 \
    --weight-decay 0.0001 \
    --scheduler cosine \
    --grad-clip 1.0 \
    --patience 12 \
    --workers 4 \
    --seed 42 \
    --slices-csv '/content/data/cache_1mm/slices_r1b_dynneg.csv' \
    --dynamic-neg \
    --keep-all-train-neg \
    --loss-params '{"bce_weight": 0.3, "dice_weight": 0.7, "pos_weight": 4.0}' \
    --run-name 'mamba_colab_500item_50epoch_R1b_NegDynamic' \
    --tag 'colab_mamba' \
    --notes 'Huấn luyện Mamba-UNet với kernel CUDA native selective_scan trên Colab' \
    --drive-sync-

In [6]:
# 9. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
import glob, json, os
from pathlib import Path

# Tự động tìm thư mục run theo RUN_NAME (nếu có) hoặc run mới nhất trong mamba_unet/runs
run_dir = None
if "RUN_NAME" in globals() and RUN_NAME:
    cand = Path("mamba_unet/runs") / RUN_NAME
    if cand.exists():
        run_dir = cand

if run_dir is None:
    local_runs = sorted(glob.glob("mamba_unet/runs/*"), key=os.path.getmtime)
    if not local_runs:
        raise FileNotFoundError("Không tìm thấy thư mục run nào trong mamba_unet/runs/")
    run_dir = Path(local_runs[-1])

ckpt_path = run_dir / "best_model.pth"
if not ckpt_path.exists():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {ckpt_path}")

# Đọc image_size, cache_dir, slices_csv, min_area từ run_config.json (fallback về biến ở Cell trên)
img_sz = globals().get("IMAGE_SIZE", 512)
eval_cache_dir = globals().get("CACHE_DIR", "/content/data/cache_1mm")
eval_slices_csv = globals().get("SLICES_CSV", "")
min_area = globals().get("MIN_TRAIN_AREA", 20.0)

rcfg_path = run_dir / "run_config.json"
if rcfg_path.exists():
    with open(rcfg_path, "r", encoding="utf-8") as f:
        rcfg = json.load(f)
    rcfg_args = rcfg.get("args", {})
    img_sz = rcfg_args.get("duneai_image_size", rcfg.get("model", {}).get("image_size", img_sz))
    eval_cache_dir = rcfg_args.get("cache_dir", eval_cache_dir)
    eval_slices_csv = rcfg_args.get("slices_csv") or eval_slices_csv
    min_area = rcfg_args.get("min_train_area", min_area)

eval_env = f"DUNEAI_IMAGE_SIZE={img_sz}"
if globals().get("FORCE_CUDA_BACKEND", True):
    eval_env += " DUNEAI_MAMBA_BACKEND=cuda"

out_csv = run_dir / "eval3d_val.csv"
out_json = run_dir / "eval3d_val.json"

print(f"🎯 Đang đánh giá 3D cho: {ckpt_path}")
print(f"📐 Image size: {img_sz} | Min area: {min_area} mm² | Cache: {eval_cache_dir}")
if eval_slices_csv:
    print(f"📄 Index CSV: {eval_slices_csv}")

eval_cmd = [
    f"{eval_env} python -u -m mamba_unet.evaluate eval",
    f'--cache-dir "{eval_cache_dir}"',
    f'--checkpoint "{ckpt_path}"',
    "--split val",
    f"--min-area {min_area}",
    f'--out-csv "{out_csv}"',
    f'--out-json "{out_json}"',
]
if eval_slices_csv and str(eval_slices_csv).strip():
    eval_cmd.append(f'--slices-csv "{str(eval_slices_csv).strip()}"')

!{" \\\n    ".join(eval_cmd)}

🎯 Đang đánh giá 3D cho: mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic/best_model.pth
📐 Image size: 448 | Min area: 5.0 mm² | Cache: /content/data/cache_1mm
📄 Index CSV: /content/data/cache_1mm/slices_r1b_dynneg.csv
🩻 ĐÁNH GIÁ 3D CẤP ĐỘ BỆNH NHÂN | Split: val (75 ca) | Thiết bị: cuda | Ngưỡng lọc u: >= 5.0 mm²
[01/75] LIDC-IDRI-0005 (12.3s) | 3D Dice: 0.3723 -> 0.3723 (+0.0000) | IoU: 0.2287 | HD95: 79.602 mm | Vol(GT/Pred): 377.5/777.5 mm³
[02/75] LIDC-IDRI-0008 (7.1s) | 3D Dice: 0.0285 -> 0.0285 (+0.0000) | IoU: 0.0144 | HD95: 53.557 mm | Vol(GT/Pred): 322.5/15132.5 mm³
[03/75] LIDC-IDRI-0022 (5.3s) | 3D Dice: 0.5772 -> 0.5774 (+0.0002) | IoU: 0.4059 | HD95: 194.657 mm | Vol(GT/Pred): 11655.0/7230.0 mm³
[04/75] LIDC-IDRI-0038 (4.5s) | 3D Dice: 0.0000 -> 0.0000 (+0.0000) | IoU: 0.0000 | HD95: 247.864 mm | Vol(GT/Pred): 52.5/412.5 mm³
[05/75] LIDC-IDRI-0042 (4.8s) | 3D Dice: 0.5539 -> 0.5595 (+0.0056) | IoU: 0.3884 | HD95: 65.57 mm | Vol(GT/Pred): 1015.0/960.0 mm³
[06/75] L

In [7]:
# 10. Lưu kết quả và Checkpoint vào Google Drive theo từng thư mục phiên chạy
# Tự động tạo mới hoặc cập nhật nối thêm (append) file REPORT.md, DATA_SPEC_AND_CONFIG.md và JSON tổng kết
import datetime, glob, json, os, shutil
from pathlib import Path

# 1. Xác định thư mục mirror Google Drive
MIRROR_SOURCES_DIR = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/DuneAI_Backups'

# 2. Tìm thư mục phiên chạy gần nhất trong mamba_unet/runs
local_runs = sorted(glob.glob("mamba_unet/runs/*"), key=os.path.getmtime)
if local_runs:
    active_local_run = Path(local_runs[-1])
    run_folder_name = active_local_run.name
else:
    run_folder_name = f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    active_local_run = Path(f"mamba_unet/runs/{run_folder_name}")
    active_local_run.mkdir(parents=True, exist_ok=True)

# 3. Thư mục đích trên Google Drive
drive_run_dir = Path(MIRROR_SOURCES_DIR) / "mamba_unet" / "runs" / run_folder_name
drive_ckpt_dir = Path(MIRROR_SOURCES_DIR) / "checkpoints"
drive_run_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

# 4. Đọc run_config.json trước để lấy đúng thông số thực tế của phiên chạy
rcfg = {}
if (active_local_run / "run_config.json").exists():
    try:
        with open(active_local_run / "run_config.json", "r", encoding="utf-8") as f:
            rcfg = json.load(f)
    except Exception:
        pass

rcfg_args = rcfg.get("args", {})
rcfg_data = rcfg.get("data", {})

# Trích xuất thông số y tế (ưu tiên run_config.json -> biến notebook -> configs/config.py)
try:
    import configs.config as med_cfg
    cfg_img_size = getattr(med_cfg, "IMAGE_SIZE", 512)
    win_width = getattr(med_cfg, "WINDOW_WIDTH", 1500)
    win_level = getattr(med_cfg, "WINDOW_LEVEL", -600)
    resamp_step = getattr(med_cfg, "RESAMPLING_STEP", 25)
    cfg_tumor_lim = getattr(med_cfg, "TUMOR_SIZE_LOW_LIM", 20.0)
    cfg_zero_ratio = getattr(med_cfg, "ZERO_SL_RATIO", 0.25)
    val_split = getattr(med_cfg, "VAL_SPLIT", 0.15)
except Exception:
    cfg_img_size, win_width, win_level, resamp_step = 512, 1500, -600, 25
    cfg_tumor_lim, cfg_zero_ratio, val_split = 20.0, 0.25, 0.15

img_size = rcfg_args.get("duneai_image_size", rcfg_data.get("image_size", globals().get("IMAGE_SIZE", cfg_img_size)))
tumor_lim = rcfg_args.get("min_train_area", rcfg_data.get("min_train_area", globals().get("MIN_TRAIN_AREA", cfg_tumor_lim)))
zero_ratio = rcfg_args.get("neg_ratio", rcfg_data.get("neg_ratio", globals().get("NEG_RATIO", cfg_zero_ratio)))
min_neg_case = rcfg_args.get("min_neg_per_case", rcfg_data.get("min_neg_per_case", globals().get("MIN_NEG_PER_CASE", 1)))
dyn_neg = rcfg_args.get("dynamic_neg", rcfg_data.get("dynamic_neg", globals().get("DYNAMIC_NEG", False)))
keep_all_neg = rcfg_args.get("keep_all_train_neg", rcfg_data.get("keep_all_train_neg", globals().get("KEEP_ALL_TRAIN_NEG", False)))

# 5. Đọc thống kê dữ liệu thực tế từ đúng file slices CSV của run
cfg_slices = rcfg_args.get("slices_csv") or rcfg_data.get("slices_csv") or globals().get("SLICES_CSV")
cfg_cache = rcfg_args.get("cache_dir") or globals().get("CACHE_DIR") or "/content/data/cache_1mm"

if cfg_slices and str(cfg_slices).strip():
    slices_path = Path(str(cfg_slices).strip())
else:
    slices_path = Path(cfg_cache) / "slices.csv"

if not slices_path.exists() and 'cache_dir' in globals() and Path(cache_dir).exists():
    candidate_sl = Path(cache_dir) / "slices.csv"
    if candidate_sl.exists():
        slices_path = candidate_sl

data_stats = {
    "num_patients": globals().get("NUM_PATIENTS", 500),
    "train_patients": "N/A",
    "val_patients": "N/A",
    "total_slices": "N/A",
    "train_slices": "N/A",
    "val_slices": "N/A",
    "pos_slices": "N/A",
    "neg_slices": "N/A",
}

if slices_path.exists():
    try:
        import pandas as pd
        df_sl = pd.read_csv(slices_path)
        data_stats["total_slices"] = int(len(df_sl))
        if "patient_id" in df_sl.columns:
            data_stats["num_patients"] = int(df_sl["patient_id"].nunique())
            if "split" in df_sl.columns:
                data_stats["train_patients"] = int(df_sl[df_sl["split"] == "train"]["patient_id"].nunique())
                data_stats["val_patients"] = int(df_sl[df_sl["split"] == "val"]["patient_id"].nunique())
        if "split" in df_sl.columns:
            data_stats["train_slices"] = int(len(df_sl[df_sl["split"] == "train"]))
            data_stats["val_slices"] = int(len(df_sl[df_sl["split"] == "val"]))
        if "has_tumor" in df_sl.columns:
            data_stats["pos_slices"] = int((df_sl["has_tumor"] == 1).sum())
            data_stats["neg_slices"] = int((df_sl["has_tumor"] == 0).sum())
    except Exception as e:
        print(f"ℹ️ Không đọc được chi tiết {slices_path.name}: {e}")

# 5.1 Lưu trữ bản sao chỉ mục slices CSV vào thư mục run (cục bộ & Drive)
if slices_path.exists():
    try:
        shutil.copy2(slices_path, active_local_run / slices_path.name)
        shutil.copy2(slices_path, drive_run_dir / slices_path.name)
        print(f"💾 Đã lưu trữ bản sao chỉ mục {slices_path.name} tại: {drive_run_dir / slices_path.name}")
    except Exception as e:
        print(f"ℹ️ Không thể sao lưu {slices_path.name}: {e}")

# 6. Đọc kết quả từ history.csv và run_config.json
now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
best_val_dice = "N/A"
best_epoch = "N/A"
best_val_iou = "N/A"
best_val_loss = "N/A"
train_dice_at_best = "N/A"
train_iou_at_best = "N/A"
train_loss_at_best = "N/A"
epochs_completed = "N/A"
duration_str = "N/A"

log_file = active_local_run / "history.csv"
if not log_file.exists():
    log_file = active_local_run / "train_log.csv"

if log_file.exists():
    try:
        import pandas as pd
        df_log = pd.read_csv(log_file)
        if len(df_log) > 0:
            res_obj = rcfg.get("result", {})
            auth_best_epoch = res_obj.get("best_epoch")
            if auth_best_epoch is not None and "epoch" in df_log.columns and (df_log["epoch"] == auth_best_epoch).any():
                best_idx = df_log.index[df_log["epoch"] == auth_best_epoch][0]
            elif "val_dice" in df_log.columns:
                best_idx = df_log["val_dice"].idxmax()
            else:
                best_idx = None
            if best_idx is not None:
                best_epoch = f"{int(df_log.loc[best_idx, 'epoch'])}"
                epochs_completed = int(df_log["epoch"].max())
                if "val_dice" in df_log.columns:
                    best_val_dice = f"{df_log.loc[best_idx, 'val_dice']:.4f}"
                if "val_iou" in df_log.columns:
                    best_val_iou = f"{df_log.loc[best_idx, 'val_iou']:.4f}"
                if "val_loss" in df_log.columns:
                    best_val_loss = f"{df_log.loc[best_idx, 'val_loss']:.4f}"
                if "train_dice" in df_log.columns:
                    train_dice_at_best = f"{df_log.loc[best_idx, 'train_dice']:.4f}"
                if "train_iou" in df_log.columns:
                    train_iou_at_best = f"{df_log.loc[best_idx, 'train_iou']:.4f}"
                if "train_loss" in df_log.columns:
                    train_loss_at_best = f"{df_log.loc[best_idx, 'train_loss']:.4f}"
    except Exception:
        pass

if "duration_sec" in rcfg:
    dur = rcfg["duration_sec"]
    duration_str = f"{int(dur // 3600):02d}h {int((dur % 3600) // 60):02d}m {int(dur % 60):02d}s ({dur:.1f}s)"
if "result" in rcfg:
    res_obj = rcfg["result"]
    if res_obj.get("best_val_dice") is not None:
        best_val_dice = f"{res_obj['best_val_dice']:.4f}"
    if res_obj.get("best_epoch") is not None:
        best_epoch = f"{res_obj['best_epoch']}"
    if epochs_completed == "N/A" and res_obj.get("epochs_completed") is not None:
        epochs_completed = int(res_obj["epochs_completed"])

# Thu thập siêu tham số
b_bone = rcfg_args.get("backbone", globals().get("BACKBONE", "mamba_vision_T"))
is_pt = rcfg_args.get("pretrained", globals().get("PRETRAINED", True))
frz_ep = rcfg_args.get("freeze_backbone_epochs", globals().get("FREEZE_BACKBONE_EPOCHS", 3))
bb_mult = rcfg_args.get("backbone_lr_mult", globals().get("BACKBONE_LR_MULT", 0.1))
act_name = rcfg_args.get("activation", globals().get("ACTIVATION", "relu"))
loss_name = rcfg_args.get("loss", globals().get("LOSS", "dice_bce"))
loss_p = rcfg_args.get("loss_params", globals().get("LOSS_PARAMS", ""))
tot_epochs = rcfg_args.get("epochs", globals().get("EPOCHS", 50))
b_size = rcfg_args.get("batch_size", globals().get("BATCH_SIZE", 8))
l_rate = rcfg_args.get("lr", globals().get("LR", 1e-4))
w_decay = rcfg_args.get("weight_decay", globals().get("WEIGHT_DECAY", 1e-4))
sched_name = rcfg_args.get("scheduler", globals().get("SCHEDULER", "cosine"))
g_clip = rcfg_args.get("grad_clip", globals().get("GRAD_CLIP", 1.0))
pat_val = rcfg_args.get("patience", globals().get("PATIENCE", 12))
num_w = rcfg_args.get("workers", globals().get("WORKERS", 4))
seed_v = rcfg_args.get("seed", globals().get("SEED", 42))
res_ckpt = rcfg_args.get("resume", globals().get("RESUME", None))
notes_v = rcfg.get("notes", globals().get("NOTES", ""))
lung_mask_v = not rcfg_args.get("no_lung_mask", globals().get("NO_LUNG_MASK", False))
augment_v = not rcfg_args.get("no_augment", globals().get("NO_AUGMENT", False))

# 7. Soạn thảo file báo cáo chi tiết REPORT.md
report_md_path = drive_run_dir / "REPORT.md"
is_resumed_session = bool(res_ckpt) and report_md_path.exists()

report_content = f"""# 📋 Báo Cáo Huấn Luyện Mamba-UNet: {run_folder_name}

- **Thời gian khởi tạo / cập nhật**: {now_str}
- **Tổng thời gian chạy**: {duration_str}
- **Trạng thái**: Hoàn tất {epochs_completed}/{tot_epochs} epochs

### 1. Phân Chia Dữ Liệu & Quy Chuẩn Tiền Xử Lý Y Tế
- **Tổng số bệnh nhân**: **{data_stats['num_patients']} ca** (Train: {data_stats['train_patients']} | Val: {data_stats['val_patients']})
- **Chiến lược phân chia**: Phân chia theo mã bệnh nhân (Patient-Level Split, Zero Leakage, Val Split: {val_split * 100:.0f}%, Seed: {seed_v})
- **File chỉ mục sử dụng**: `{slices_path}`
- **Thống kê lát cắt CT**:
  - **Tổng số lát cắt giữ lại**: **{data_stats['total_slices']} lát** (Train: {data_stats['train_slices']} | Val: {data_stats['val_slices']})
  - **Lát dương tính (chứa u)**: **{data_stats['pos_slices']} lát**
  - **Lát âm tính (không u, có phổi)**: **{data_stats['neg_slices']} lát** (`neg_ratio={zero_ratio}`, `min_neg_per_case={min_neg_case}`, `dynamic_neg={dyn_neg}`, `keep_all_train_neg={keep_all_neg}`)
- **Quy chuẩn lọc khối u**: `min_train_area = {tumor_lim} mm²` (loại bỏ u nhỏ hơn {tumor_lim} mm²)
- **Chuẩn hóa Hounsfield Unit (HU)**:
  - Cửa sổ phổi: `WINDOW_WIDTH = {win_width} HU`, `WINDOW_LEVEL = {win_level} HU` (Dải quan sát: [{win_level - win_width//2}, {win_level + win_width//2}] HU)
  - Lượng tử hóa bước nhảy: `RESAMPLING_STEP = {resamp_step} HU` (làm tròn {resamp_step} HU mỗi bước để giảm nhiễu hạt máy quét CT và nén dữ liệu)
- **Định dạng đầu vào**: 2.5D (ghép 3 lát cắt kế tiếp [z-1, z, z+1], kích thước {img_size}x{img_size})
- **Áp dụng mặt nạ phổi (Lung Mask)**: {lung_mask_v} | **Tăng cường dữ liệu (Augmentation)**: {augment_v}

### 2. Kiến Trúc Mô Hình & Tối Ưu Hóa
- **Kiến trúc**: Mamba-UNet (NVIDIA MambaVision Encoder + Symmetric UNet Decoder)
  - **Backbone**: `{b_bone}` (Pretrained ImageNet-1K: `{is_pt}`)
  - **Lịch đóng băng Backbone**: {frz_ep} epochs đầu (Tỷ lệ LR Backbone: {bb_mult})
  - **Decoder Activation**: `{act_name}`
- **Hàm mất mát (Loss)**: `{loss_name}` (Tham số: `{loss_p}`)
- **Siêu tham số tối ưu**:
  - **Optimizer**: AdamW (Base LR: `{l_rate}`, Backbone LR: `{l_rate * bb_mult}`, Weight Decay: `{w_decay}`)
  - **LR Scheduler**: `{sched_name}`
  - **Gradient Clip**: `{g_clip}` | **Patience (Early Stopping)**: `{pat_val}`
  - **DataLoader**: Batch Size = {b_size} | Workers = {num_w} | Seed = {seed_v}
- **Backend Mamba**: CUDA Native Selective Scan (`mamba_ssm` + `causal_conv1d`)

### 3. Kết Quả Huấn Luyện (Training Results)
- **Epoch tốt nhất (Best Epoch)**: **Epoch {best_epoch}**
  - **Best Val Dice Score**: **{best_val_dice}** ⭐ [BEST]
  - **Best Val IoU (Jaccard)**: **{best_val_iou}**
  - **Val Loss**: **{best_val_loss}**
  - **Train Dice tại Best Epoch**: `{train_dice_at_best}` | **Train IoU**: `{train_iou_at_best}` | **Train Loss**: `{train_loss_at_best}`
- **Ghi chú**: {notes_v}
"""

if is_resumed_session:
    print(f"🔄 Đang nối thêm (append) thông tin phiên Resume vào: {report_md_path}")
    resume_block = f"\n---\n### 🔄 Phiên Tiếp Tục Huấn Luyện (Resume Session) - {now_str}\n" + report_content
    with open(report_md_path, "a", encoding="utf-8") as f:
        f.write(resume_block)
else:
    print(f"📝 Đang tạo mới file báo cáo tổng hợp: {report_md_path}")
    with open(report_md_path, "w", encoding="utf-8") as f:
        f.write(report_content)

# Lưu bản sao REPORT.md vào thư mục cục bộ và sao chép thành DATA_SPEC_AND_CONFIG.md
shutil.copy2(report_md_path, active_local_run / "REPORT.md")
shutil.copy2(report_md_path, drive_run_dir / "DATA_SPEC_AND_CONFIG.md")
shutil.copy2(report_md_path, active_local_run / "DATA_SPEC_AND_CONFIG.md")

# 8. Xuất file JSON toàn diện (training_report.json)
report_json_data = {
    "run_folder": run_folder_name,
    "timestamp": now_str,
    "arch": "mamba_unet",
    "backbone": b_bone,
    "pretrained": is_pt,
    "freeze_backbone_epochs": frz_ep,
    "backbone_lr_mult": bb_mult,
    "activation": act_name,
    "loss": loss_name,
    "loss_params": loss_p,
    "epochs_target": tot_epochs,
    "epochs_completed": epochs_completed,
    "batch_size": b_size,
    "lr": l_rate,
    "weight_decay": w_decay,
    "scheduler": sched_name,
    "workers": num_w,
    "best_val_dice": best_val_dice,
    "best_val_iou": best_val_iou,
    "best_val_loss": best_val_loss,
    "best_epoch": best_epoch,
    "duration": duration_str,
    "resume": res_ckpt,
    "notes": notes_v,
    "data_specification": {
        "slices_csv": str(slices_path),
        "num_patients": data_stats["num_patients"],
        "train_patients": data_stats["train_patients"],
        "val_patients": data_stats["val_patients"],
        "total_slices": data_stats["total_slices"],
        "train_slices": data_stats["train_slices"],
        "val_slices": data_stats["val_slices"],
        "pos_slices": data_stats["pos_slices"],
        "neg_slices": data_stats["neg_slices"],
        "min_train_area_mm2": tumor_lim,
        "resampling_step_hu": resamp_step,
        "window_width_hu": win_width,
        "window_level_hu": win_level,
        "neg_ratio": zero_ratio,
        "min_neg_per_case": min_neg_case,
        "dynamic_neg": dyn_neg,
        "keep_all_train_neg": keep_all_neg,
        "val_split": val_split,
        "image_size": img_size,
        "lung_mask": lung_mask_v,
        "augmentation": augment_v,
    }
}
with open(drive_run_dir / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(report_json_data, f, indent=2, ensure_ascii=False)
shutil.copy2(drive_run_dir / "training_report.json", active_local_run / "training_report.json")

# 9. Sao chép toàn bộ checkpoints và artifacts của phiên chạy vào Drive
print(f"🚀 Đang sao lưu toàn bộ files từ {active_local_run} sang {drive_run_dir}...")
!cp -ru {active_local_run}/* '{drive_run_dir}/' 2>/dev/null || true
!cp -ru checkpoints/* '{drive_ckpt_dir}/' 2>/dev/null || true

print("=" * 80)
print(f"✅ HOÀN TẤT SAO LƯU!")
print(f"📁 Thư mục phiên chạy tại Google Drive: {drive_run_dir}")
print(f"📄 Báo cáo chi tiết: {report_md_path}")
print(f"📊 Hồ sơ cấu hình dữ liệu: {drive_run_dir / 'DATA_SPEC_AND_CONFIG.md'}")
print(f"🎯 Google Drive Desktop sẽ tự động đồng bộ thư mục này về máy tính của bạn!")
print("=" * 80)

💾 Đã lưu trữ bản sao chỉ mục slices_r1b_dynneg.csv tại: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic/slices_r1b_dynneg.csv
📝 Đang tạo mới file báo cáo tổng hợp: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic/REPORT.md
🚀 Đang sao lưu toàn bộ files từ mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic sang /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic...
✅ HOÀN TẤT SAO LƯU!
📁 Thư mục phiên chạy tại Google Drive: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic
📄 Báo cáo chi tiết: /content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/mamba_unet/runs/mamba_colab_500item_50epoch_R1b_NegDynamic/REPORT.md
📊 Hồ sơ cấu

In [ ]:
# Lệnh tự động tắt máy SAU KHI huấn luyện xong
# from google.colab import runtime
# runtime.unassign()